In [6]:
def pull_ds_index_fut(db, mnemonic, start, end):
    sql = f"""
        SELECT datadate, pi
        FROM trsamp_dsfut.dsfut
        WHERE dsmnemonic = '{mnemonic.upper()}'
          AND datadate BETWEEN '{start}' AND '{end}'
        ORDER BY datadate
    """
    df = db.raw_sql(sql, date_cols=['datadate'])
    if df.empty:
        return pd.Series(dtype=float, name=mnemonic)
    return df.set_index('datadate')['pi'].dropna()

for mnemonic in ['DJUBSTR', 'DJAIGTR', 'GSCITOT', 'DJUBS', 'SPGCCITR']:
    try:
        s = pull_ds_index_fut(db, mnemonic, '1990-01-01', '2023-12-31')
        print(f'{mnemonic}: {len(s)} obs, {s.index[0].date() if len(s)>0 else "empty"} to {s.index[-1].date() if len(s)>0 else "empty"}')
    except Exception as e:
        print(f'{mnemonic}: failed — {e}')

DJUBSTR: failed — (psycopg2.errors.UndefinedTable) relation "trsamp_dsfut.dsfut" does not exist
LINE 3:         FROM trsamp_dsfut.dsfut
                     ^

[SQL: 
        SELECT datadate, pi
        FROM trsamp_dsfut.dsfut
        WHERE dsmnemonic = 'DJUBSTR'
          AND datadate BETWEEN '1990-01-01' AND '2023-12-31'
        ORDER BY datadate
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)
DJAIGTR: failed — (psycopg2.errors.UndefinedTable) relation "trsamp_dsfut.dsfut" does not exist
LINE 3:         FROM trsamp_dsfut.dsfut
                     ^

[SQL: 
        SELECT datadate, pi
        FROM trsamp_dsfut.dsfut
        WHERE dsmnemonic = 'DJAIGTR'
          AND datadate BETWEEN '1990-01-01' AND '2023-12-31'
        ORDER BY datadate
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)
GSCITOT: failed — (psycopg2.errors.UndefinedTable) relation "trsamp_dsfut.dsfut" does not exist
LINE 3:         FROM trsamp_dsfut.dsfut
                     ^

[SQL

In [16]:
import wrds
db = wrds.Connection(wrds_username='fdxavierj5')

Loading library list...
Done


In [17]:
gsci = db.raw_sql("""
    SELECT date_, close_
    FROM trsamp_dscom.wrds_cmdy_data
    WHERE comcode = 3335
      AND close_ IS NOT NULL
    ORDER BY date_
""", date_cols=['date_'])

print(f'Rows: {len(gsci)}')
print(f'From: {gsci["date_"].iloc[0].date()} to {gsci["date_"].iloc[-1].date()}')

# Resample to Wednesday weekly and compute returns
gsci_s = gsci.set_index('date_')['close_']
gsci_w = gsci_s.resample('W-WED').last()
gsci_ret = gsci_w.pct_change().dropna()
gsci_ret.name = 'GSCI'

gsci_ret.to_csv('bcom_weekly.csv', header=True)
print(f'\nSaved as bcom_weekly.csv (using GSCI as BCOM proxy):')
print(f'  T={len(gsci_ret)}, {gsci_ret.index[0].date()} to {gsci_ret.index[-1].date()}')
print(f'  Mean={gsci_ret.mean()*52*100:.2f}%pa  Std={gsci_ret.std()*np.sqrt(52)*100:.2f}%pa')

Rows: 3219
From: 2007-01-02 to 2020-11-17

Saved as bcom_weekly.csv (using GSCI as BCOM proxy):
  T=724, 2007-01-10 to 2020-11-18
  Mean=-4.43%pa  Std=25.26%pa


C:\Users\fdxja\AppData\Local\Temp\ipykernel_40212\3821751006.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  gsci_ret = gsci_w.pct_change().dropna()
